# Envío de clasificación por Batch API con tu modelo fine-tuned

Esta notebook prepara un archivo `.jsonl`, lo sube, crea el batch, consulta su estado y descarga los resultados para unirlos con tu CSV original.

In [1]:
# Si hace falta:
# !pip install -q openai pandas tqdm

In [2]:
import os
import json
import time
from typing import Any, Dict, Optional

import pandas as pd
from openai import OpenAI

## Configuración

In [23]:
MODEL_NAME = 'ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs'
INPUT_CSV = 'DNU_data/muestra_100.csv'   # luego puedes cambiarlo por uno más grande
TEXT_COLUMN = None
MAX_ROWS = None                 # None = usar todas las filas
REQUESTS_JSONL = 'batch_requests.jsonl'
RESULTS_JSONL = 'batch_results.jsonl'
OUTPUT_CSV = 'outputs/muestra_con_batch_gpt.csv'

# Ventana admitida por Batch API
COMPLETION_WINDOW = '24h'

# Tarifas estimadas para controlar presupuesto.
# Cámbialas si quieres usar otros valores para tu propia cuenta/modelo.
PRICE_INPUT_PER_1M_SYNC = 3.00
PRICE_OUTPUT_PER_1M_SYNC = 6.00

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('No encuentro OPENAI_API_KEY en variables de entorno.')

## Cargar dataset

In [4]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas totales: {len(df)}')
print('Columnas:')
print(list(df.columns))
df.head(3)

Filas totales: 100
Columnas:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels', 'HATEFUL']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels,HATEFUL
0,Sat Mar 06 23:16:38 +0000 2021,1368339770686996480,1368339770686996480,Extranjeros participan de los destrozos realiz...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0
1,Fri Mar 05 17:33:11 +0000 2021,1367890949792272391,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367887e+18,1.367887e+18,142093384.0,NaN,...,0,0,0,0,0,0,0,[],0,0
2,Fri Mar 05 22:02:06 +0000 2021,1367958627739258880,1367958627739258880,La diferencia con los kirchneristas ya no es s...,"<a href=""http://twitter.com/download/iphone"" r...",True,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0


In [5]:
text_col = 'text'
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)

print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

Columna de texto: text
Filas a procesar: 100


## Prompt mínimo

La idea es mantener el request corto para ahorrar tokens.

In [6]:
SYSTEM_MESSAGE = ''
USER_TEMPLATE = '{{"labels": []}}\nTexto:\n{text}'

## Estimación rápida de costo y control de presupuesto

Con tus mediciones previas por request, la API Batch cobra 50% menos que la API sincrónica. La guía y el FAQ oficial indican ese descuento y una ventana de hasta 24 horas. Puedes usar esta celda para ver si 20 USD te alcanzan, antes de subir nada.

In [7]:
# Tus números observados en una prueba real:
OBS_PROMPT_TOKENS = 231
OBS_COMPLETION_TOKENS = 8

n = len(work_df)

sync_input_cost = (OBS_PROMPT_TOKENS * n) / 1_000_000 * PRICE_INPUT_PER_1M_SYNC
sync_output_cost = (OBS_COMPLETION_TOKENS * n) / 1_000_000 * PRICE_OUTPUT_PER_1M_SYNC
sync_total = sync_input_cost + sync_output_cost

batch_total = sync_total * 0.5

print('Estimación SINCRÓNICA:', round(sync_total, 2), 'USD')
print('Estimación BATCH (50% off):', round(batch_total, 2), 'USD')
print('¿Alcanzan 20 USD?', batch_total <= 20)

Estimación SINCRÓNICA: 0.07 USD
Estimación BATCH (50% off): 0.04 USD
¿Alcanzan 20 USD? True


## Preparar archivo JSONL de requests

Cada línea del archivo representa una request independiente al endpoint `/v1/chat/completions`.

In [8]:
def make_request_line(row_id: int, text: str) -> Dict[str, Any]:
    messages = []
    if SYSTEM_MESSAGE.strip():
        messages.append({'role': 'system', 'content': SYSTEM_MESSAGE})
    messages.append({'role': 'user', 'content': USER_TEMPLATE.format(text=text)})

    return {
        'custom_id': f'row-{row_id}',
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': MODEL_NAME,
            'temperature': 0,
            'max_tokens': 40,
            'messages': messages
        }
    }

with open(REQUESTS_JSONL, 'w', encoding='utf-8') as f:
    for idx, row in work_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ''
        line = make_request_line(idx, text)
        f.write(json.dumps(line, ensure_ascii=False) + '\n')

print(f'Archivo creado: {REQUESTS_JSONL}')

Archivo creado: batch_requests.jsonl


## Subir archivo y crear el batch

In [9]:
input_file = client.files.create(
    file=open(REQUESTS_JSONL, 'rb'),
    purpose='batch'
)

print('input_file_id:', input_file.id)

batch = client.batches.create(
    input_file_id=input_file.id,
    endpoint='/v1/chat/completions',
    completion_window=COMPLETION_WINDOW,
    metadata={'job': 'hate-classification'}
)

print('batch_id:', batch.id)
print('status:', batch.status)
batch

input_file_id: file-GQ3bnAkhVnJn4gZ67yKCbB
batch_id: batch_69d02c04edec8190b6a577c1fe848a81
status: validating


Batch(id='batch_69d02c04edec8190b6a577c1fe848a81', completion_window='24h', created_at=1775250436, endpoint='/v1/chat/completions', input_file_id='file-GQ3bnAkhVnJn4gZ67yKCbB', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1775336836, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'job': 'hate-classification'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

## Consultar estado del batch

In [20]:
# Reemplaza por el id devuelto si vuelves en otro momento:
BATCH_ID = batch.id

status = client.batches.retrieve(BATCH_ID)
print('status:', status.status)
status

status: completed


Batch(id='batch_69d02c04edec8190b6a577c1fe848a81', completion_window='24h', created_at=1775250436, endpoint='/v1/chat/completions', input_file_id='file-GQ3bnAkhVnJn4gZ67yKCbB', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1775251573, error_file_id=None, errors=None, expired_at=None, expires_at=1775336836, failed_at=None, finalizing_at=1775251568, in_progress_at=1775250499, metadata={'job': 'hate-classification'}, model='ft:gpt-3.5-turbo-0125:personal:hatemultiv4:9uouAqOs', output_file_id='file-HtSjzgXs2rJZUjs7jrYZoR', request_counts=BatchRequestCounts(completed=100, failed=0, total=100), usage=BatchUsage(input_tokens=5279, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=85, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=5364))

## Polling simple hasta completar

La Batch API devuelve resultados dentro de 24 horas y a menudo antes.

In [16]:
BATCH_ID = batch.id
POLL_EVERY_SECONDS = 30

while True:
    status = client.batches.retrieve(BATCH_ID)
    print(time.strftime('%H:%M:%S'), 'status =', status.status)
    if status.status in {'completed', 'failed', 'expired', 'cancelled'}:
        break
    time.sleep(POLL_EVERY_SECONDS)

status

18:13:34 status = in_progress


KeyboardInterrupt: 

## Descargar archivo de resultados

In [21]:
if status.status == 'completed':
    output_file_id = status.output_file_id
    print('output_file_id:', output_file_id)

    content = client.files.content(output_file_id)
    with open(RESULTS_JSONL, 'wb') as f:
        f.write(content.read())

    print(f'Resultados guardados en: {RESULTS_JSONL}')
else:
    print('El batch no terminó en estado completed. Revisa status.error_file_id y status.errors si existen.')

output_file_id: file-HtSjzgXs2rJZUjs7jrYZoR
Resultados guardados en: batch_results.jsonl


## Parsear resultados y unirlos al CSV original

In [24]:
def safe_json_loads(text: str) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            return None
    return None

rows = []
with open(RESULTS_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        custom_id = obj.get('custom_id')

        raw_content = None
        prompt_tokens = None
        completion_tokens = None
        total_tokens = None
        status_code = None
        error_msg = None

        response = obj.get('response')
        if response:
            status_code = response.get('status_code')
            body = response.get('body', {})
            usage = body.get('usage', {})
            prompt_tokens = usage.get('prompt_tokens')
            completion_tokens = usage.get('completion_tokens')
            total_tokens = usage.get('total_tokens')

            choices = body.get('choices', [])
            if choices and isinstance(choices, list):
                raw_content = choices[0].get('message', {}).get('content')

        if obj.get('error'):
            error_msg = str(obj.get('error'))

        parsed = safe_json_loads(raw_content)
        labels = None
        if isinstance(parsed, dict):
            labels = parsed.get('labels')

        row_id = int(custom_id.replace('row-', '')) if custom_id and custom_id.startswith('row-') else None

        rows.append({
            'row_id': row_id,
            'batch_raw_response': raw_content,
            'batch_labels': json.dumps(labels, ensure_ascii=False) if labels is not None else None,
            'batch_prompt_tokens': prompt_tokens,
            'batch_completion_tokens': completion_tokens,
            'batch_total_tokens': total_tokens,
            'batch_status_code': status_code,
            'batch_error': error_msg
        })

res_df = pd.DataFrame(rows).sort_values('row_id')
final_df = work_df.reset_index().rename(columns={'index': 'row_id'}).merge(res_df, on='row_id', how='left')
final_df.to_csv(OUTPUT_CSV, index=False)

print(f'CSV final guardado en: {OUTPUT_CSV}')
final_df.head(10)

CSV final guardado en: outputs/muestra_con_batch_gpt.csv


,row_id,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,...,labels,n_labels,HATEFUL,batch_raw_response,batch_labels,batch_prompt_tokens,batch_completion_tokens,batch_total_tokens,batch_status_code,batch_error
0,0,Sat Mar 06 23:16:38 +0000 2021,1368339770686996480,1368339770686996480,Extranjeros participan de los destrozos realiz...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,...,[],0,0,,None,51,0,51,200,None
1,1,Fri Mar 05 17:33:11 +0000 2021,1367890949792272391,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367887e+18,1.367887e+18,1.420934e+08,...,[],0,0,,None,60,0,60,200,None
2,2,Fri Mar 05 22:02:06 +0000 2021,1367958627739258880,1367958627739258880,La diferencia con los kirchneristas ya no es s...,"<a href=""http://twitter.com/download/iphone"" r...",True,NaN,NaN,NaN,...,[],0,0,,None,54,0,54,200,None
3,3,Sun Mar 07 14:26:52 +0000 2021,1368568840351870976,1368568840351870976,@Lady_Chiyome La dictadura de pelusones del 73...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.368373e+18,1.368373e+18,1.360764e+18,...,[],0,0,,None,62,0,62,200,None
4,4,Sun Mar 07 02:07:27 +0000 2021,1368382756502306816,1368382756502306816,@Mari21Llll La pinché Momia solo es un florero...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.368354e+18,1.368354e+18,1.274648e+18,...,[],0,0,,None,62,0,62,200,None
5,5,Fri Mar 05 11:33:53 +0000 2021,1367800528059707392,1367800528059707392,El Gobierno derogó el decreto de Mauricio Macr...,"<a href=""https://zapier.com/"" rel=""nofollow"">Z...",True,NaN,NaN,NaN,...,[],0,0,,None,56,0,56,200,None
6,6,Sat Mar 06 12:45:50 +0000 2021,1368181023754887170,1368181023754887170,@MikeHam060 Fueron expulsados por tener antece...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.368175e+18,1.368175e+18,1.102314e+18,...,[],0,0,,None,55,0,55,200,None
7,7,Fri Mar 05 19:20:32 +0000 2021,1367917965572775943,1367917965572775943,@bastogne1943 Exacto... Hoy los únicos capaces...,"<a href=""http://twitter.com/download/android"" ...",True,1.367881e+18,1.367881e+18,1.072829e+18,...,[],0,0,,None,63,0,63,200,None
8,8,Fri Mar 05 17:18:08 +0000 2021,1367887162449002504,1367887162449002504,@rubenprofe @Elputonisa @ArielSanche2 @Dasmatn...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.367887e+18,1.367887e+18,1.500477e+08,...,[],0,0,,None,68,0,68,200,None
9,9,Fri Mar 05 23:28:20 +0000 2021,1367980325494734849,1367980325494734849,@neuquen24horas @alferdez @alferdezprensa siem...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367850e+18,1.367850e+18,7.575446e+08,...,[],0,0,,None,64,0,64,200,None


## Resumen de clasificaciones obtenidas

In [27]:
import ast

# Parsear batch_labels (JSON string -> lista)
final_df["_labels_list"] = final_df["batch_labels"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else []
)

CATS = ["CALLS", "WOMEN", "LGBTI", "RACISM", "CLASS", "POLITICS", "DISABLED", "APPEARANCE", "CRIMINAL"]

total = len(final_df)
errores = final_df["batch_error"].notna().sum()
sin_labels = (final_df["_labels_list"].apply(len) == 0).sum()
con_labels = (final_df["_labels_list"].apply(len) > 0).sum()

print("=" * 50)
print("RESUMEN DE CLASIFICACIÓN BATCH")
print("=" * 50)
print(f"  Total filas:          {total:>6}")
print(f"  Con errores:          {errores:>6}")
print(f"  Sin etiqueta (ok):    {sin_labels:>6}  ({sin_labels/total*100:.1f}%)")
print(f"  Con alguna etiqueta:  {con_labels:>6}  ({con_labels/total*100:.1f}%)")
print()

# Frecuencia por categoría
freq = {cat: final_df["_labels_list"].apply(lambda l: cat in l).sum() for cat in CATS}
freq_df = (
    pd.DataFrame.from_dict(freq, orient="index", columns=["n"])
    .assign(pct=lambda x: (x["n"] / total * 100).round(2))
    .sort_values("n", ascending=False)
)
print("Frecuencia por categoría:")
display(freq_df[freq_df["n"] > 0])

# Combinaciones más frecuentes
combos = (
    final_df["_labels_list"]
    .apply(lambda l: tuple(sorted(l)) if l else ("(sin etiqueta)",))
    .value_counts()
    .head(10)
    .reset_index()
)
combos.columns = ["combinacion", "n"]
combos["pct"] = (combos["n"] / total * 100).round(2)
print("Top 10 combinaciones de etiquetas:")
display(combos)

final_df.drop(columns=["_labels_list"], inplace=True)


RESUMEN DE CLASIFICACIÓN BATCH
  Total filas:             100
  Con errores:               0
  Sin etiqueta (ok):       100  (100.0%)
  Con alguna etiqueta:       0  (0.0%)

Frecuencia por categoría:


,n,pct


Top 10 combinaciones de etiquetas:


,combinacion,n,pct
0,"((sin etiqueta),)",100,100.0
